# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides an interactive guide for loading, exploring, and analyzing the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and record sets from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the URL for the Croissant schema
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the croissant dataset
dataset = mlc.Dataset(croissant_url)
# Print a high-level summary
dataset_metadata = dataset.metadata
print(f"Title: {dataset_metadata.name}")
print(f"Description: {dataset_metadata.description}")
print(f"Version: {dataset_metadata.version}")
print(f"Authors: {[getattr(a,'name',getattr(a,'@id',None)) for a in getattr(dataset_metadata,'author',[])]}")


## 2. Data Overview

Review available record sets, their `@id` values, and the available fields and columns in the dataset. 

We always refer to record sets, fields, and columns by their `@id` for precise and reproducible referencing.

In [ ]:
# List all record sets with their @id and field info
print("Available record sets in the dataset:")
record_sets = list(dataset.metadata.record_sets)
record_set_ids = []
for record_set in record_sets:
    print(f"- RecordSet name: {record_set.name}")
    print(f"  @id         : {record_set['@id']}")
    field_ids = [field['@id'] for field in getattr(record_set, 'fields', [])]
    print(f"  Fields      : {field_ids if field_ids else 'No explicit fields listed'}")
    record_set_ids.append(record_set['@id'])
    print()
# Example: Print a sample record from each record set
for rs in record_sets:
    print(f"Sample record from {rs['@id']}")
    for i, record in enumerate(dataset.records(record_set=rs['@id'])):
        print(json.dumps(record, indent=2) if i < 1 else "")
        break


## 3. Data Extraction

Let's load the records for each record set into pandas DataFrames. This will facilitate downstream exploration and analytics.

All extraction, filtering, and transformation steps are fielded by the relevant Croissant `@id` values.

In [ ]:
# Load all record sets into pandas DataFrames keyed by their @id
dataframes = {}

for rs in record_sets:
    rs_id = rs['@id']
    records = list(dataset.records(record_set=rs_id))
    if records:
        try:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded DataFrame for record set {rs_id} with {df.shape[0]} rows and columns: {df.columns.tolist()}")
        except Exception as e:
            print(f"Could not convert records from {rs_id} to DataFrame: {e}")
    else:
        print(f"No records found for record set {rs_id}.")
# List available record set IDs for further analysis
print("\nAvailable record set IDs:")
print(list(dataframes.keys()))

## 4. Exploratory Data Analysis (EDA)

Let's select a record set to explore and process. For demonstration, we'll select the first loaded record set (replace with a different `@id` if desired).

- We will filter records based on a numeric field (referenced by its `@id`/DataFrame column name).
- We'll also normalize this field and (optionally) group by another field.

In [ ]:
# Choose a record set (by @id) for EDA
selected_record_set_id = next(iter(dataframes))
df = dataframes[selected_record_set_id]
print(f"Exploring record set: {selected_record_set_id}")
print("Columns available:", df.columns.tolist())

# Try to infer a suitable numeric field. You may change this to any field @id from your dataset.
numeric_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
if not numeric_candidates:
    # Try to cast eligible columns to numeric
    for c in df.columns:
        try:
            df[c] = pd.to_numeric(df[c])
            if pd.api.types.is_numeric_dtype(df[c]):
                numeric_candidates.append(c)
        except:
            continue

if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
    print(f"Using numeric field for EDA: {numeric_field_id}")
else:
    raise ValueError("No numeric field found for EDA.")

threshold = df[numeric_field_id].mean() if not df[numeric_field_id].isnull().all() else 0

filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with '{numeric_field_id}' > {threshold:.2f} (total: {filtered_df.shape[0]}):")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized '{numeric_field_id}' (first few rows):")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Optionally group by a categorical field (pick the first object-type or string field)
group_candidates = [c for c in df.columns if df[c].dtype == object and c != numeric_field_id]
if group_candidates:
    group_field_id = group_candidates[0]
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped '{numeric_field_id}' mean by '{group_field_id}':")
    print(grouped_df.head())
else:
    print("No suitable categorical field to group by.")

## 5. Visualization

Let us visualize the distribution of the selected numeric field and, if available, its relationship with the categorical grouping.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

# Histogram of numeric field
plt.figure(figsize=(8,4))
filtered_df[numeric_field_id].hist(bins=30)
plt.title(f"Histogram of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If grouped, bar plot means
if 'grouped_df' in locals():
    plt.figure(figsize=(10, 4))
    plt.bar(grouped_df[group_field_id], grouped_df[numeric_field_id])
    plt.title(f"Mean of {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we've demonstrated using the `mlcroissant` library to load and analyze the FAIR² protein dataset through its Croissant schema. We programmatically referenced record and field elements with their `@id` for reproducibility.

With this workflow, you can extend the exploration to other record sets or fields and apply advanced processing or modeling as required.